# Dynamic levels — three detectors

An explorer for horizontal support / resistance levels. It detects, inspects and visualizes levels from three interchangeable detectors and compares them side by side. Research / visualization only — no backtest, no P&L.

The detector is a single config field (level_detector on LevelParams), chosen like any other knob, so the same code path powers all three sections and the live strategies. Data is loaded once and configured centrally in the Configuration chapter; each detector section only swaps the detector and its own knobs.

The three detectors:
- pivot_level — pivot-seeded levels, tracked until invalidated by a touch or a bracket. The only detector with a pullback family.
- cluster_level — merges nearby pivots into one level per price zone; a level strengthens on touches and dies only on a decisive close-through break.
- touch_level — a zone becomes a level only once it has been touched enough times (significance by frequency); median-clustered.


## Contents

- [Configuration](#Configuration)
- [Pivot level](#Pivot-level)
- [Cluster level](#Cluster-level)
- [Touch level](#Touch-level)


## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

### Automatic

Project-wide defaults from the configurators. Override any of them in the manual cell below.

In [ ]:
import time, dataclasses
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

# Candles come from the shared cache via the global ACTIVE spec
# (engine/data_configurator.py). The detector + its knobs live on LevelParams
# (engine/strategy_configurator.py); detect_levels dispatches on level_detector.
from engine.data_configurator import ACTIVE, load_data, LIVE_DIR
from engine.levels import detect_levels, LEVEL_SOURCE_NAMES
from engine.strategy_configurator import params_for
from engine.visualization import plot_levels
from engine.indicators import ema

DATA_CONFIG = ACTIVE                                # data handle; override below
BASE_LEVEL_PARAMS = params_for("level_breakout")    # LevelParams defaults (shared knobs)
print("Detectors available:", LEVEL_SOURCE_NAMES)

### Manual

Per-notebook overrides on top of the automatic config. DATA_OVERRIDES tunes the data window; LEVEL_OVERRIDES tunes the shared detector knobs (pivot window, ATR period, tolerance) that all three detectors read. The per-detector knobs (cluster_ and touch_ prefixes) are set in each section's Parameters cell. Leave the dicts empty to stay automatic.

In [ ]:
# manual override - DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}      # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

# manual override - SHARED detector knobs (apply to all three detectors).
LEVEL_OVERRIDES = {}     # e.g. {"level_pivot_window": 3, "level_delta_mode": "atr", "level_delta": 0.5}
BASE_LEVEL_PARAMS = dataclasses.replace(BASE_LEVEL_PARAMS, **LEVEL_OVERRIDES)

### Final configuration

Load the candles and define the shared helpers every section reuses.

In [ ]:
df = load_data(DATA_CONFIG)
print(f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m - {len(df)} bars "
      f"from {df.index[0]} to {df.index[-1]}")
df.head()

In [ ]:
# Shared helpers - each detector section calls these so the sections differ only
# by the detector and its knobs.
EMA_SPANS = [20, 50, 200]                      # EMA periods overlaid (causal: bar i uses bars 0..i)
_EMA_COLORS = ["#3b82f6", "#f97316", "#a855f7", "#14b8a6", "#eab308"]

def print_level_counts(levels):
    for kind, lst in levels.items():
        active = sum(1 for l in lst if l.invalidated_at is None)
        print(f"{kind:11}: {len(lst):4d} total | {active:4d} active | {len(lst)-active:4d} invalidated")

def levels_table(levels):
    """Every level as a sortable row: seed / confirmation / invalidation bars,
    lifespan and strength. lifespan_candles = (invalidation or last bar) - seed."""
    last_idx = len(df) - 1
    rows = []
    for kind, lst in levels.items():
        for lv in lst:
            end_idx = lv.invalidated_at if lv.invalidated_at is not None else last_idx
            rows.append({
                "kind": kind,
                "price": round(lv.price, 2),
                "seed_ts": df.index[lv.start_idx],
                "confirmed_ts": df.index[lv.confirmed_idx],
                "invalidated_ts": df.index[lv.invalidated_at] if lv.invalidated_at is not None else None,
                "lifespan_candles": end_idx - lv.start_idx,
                "strength": round(lv.strength, 2),
                "active": lv.invalidated_at is None,
            })
    return pd.DataFrame(rows).sort_values(["kind", "seed_ts"]).reset_index(drop=True)

def plot_levels_emas(levels, title):
    fig = plot_levels(df, levels, show_invalidated=False, title=title)
    for i, span in enumerate(EMA_SPANS):
        fig.add_trace(go.Scatter(x=df.index, y=ema(df["close"], span), mode="lines",
                                 line=dict(color=_EMA_COLORS[i % len(_EMA_COLORS)], width=1.3),
                                 name=f"EMA {span}"))
    return fig

def run_live_levels(cfg, poll_seconds=30):
    """Auto-refreshing browser chart of this detector's live level map. Plots only -
    no orders. Refetches each poll (drops the still-forming bar) and re-detects with
    cfg. Interrupt the cell to stop."""
    chart_path = LIVE_DIR / f"{DATA_CONFIG.symbol}_{DATA_CONFIG.interval}_levels_{cfg.level_detector}.html"
    chart_path.parent.mkdir(parents=True, exist_ok=True)
    def _write(_df, _levels):
        fig = plot_levels_emas(_levels, f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | LIVE "
                               f"{cfg.level_detector} - {_df.index[-1]:%Y-%m-%d %H:%M} UTC")
        fig.write_html(str(chart_path))
        html = chart_path.read_text()
        chart_path.write_text(html.replace("<head>", f'<head><meta http-equiv="refresh" content="{poll_seconds}">', 1))
    uri = chart_path.resolve().as_uri()
    display(HTML(f'<a href="{uri}" target="_blank" rel="noopener">Open live {cfg.level_detector} chart - '
                 f'{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m</a>'))
    print(f"Live {cfg.level_detector} preview -> {uri}\nAuto-refreshes every {poll_seconds}s. Interrupt to stop.")
    try:
        while True:
            live_df = load_data(DATA_CONFIG, refresh=True).iloc[:-1]   # drop the still-forming bar
            globals()["df"] = live_df                                   # so the helpers index the live frame
            _write(live_df, detect_levels(live_df, cfg))
            print(f"updated {live_df.index[-1]} UTC | {len(live_df)} bars | next in {poll_seconds}s - interrupt to stop.")
            time.sleep(poll_seconds)
    except KeyboardInterrupt:
        print("live preview stopped.")

## Pivot level

Pivot-seeded levels, each tracked from a strict symmetric pivot until invalidated. A pivot at bar i is confirmed only at i + pivot_window, so a level never depends on future bars. Three families:

- Resistance (red) - seeded at minimum lows; dies when a later candle's high comes within tolerance, or it is bracketed invalidation_candles times.
- Support (green) - seeded at maximum highs; dies on a later low within tolerance, or bracketing.
- Pullback (orange) - seeded at minimum highs or maximum lows; an inside bar seeds two. pivot_level is the only detector with a pullback family.

A single touch within tolerance retires a level here - that is the trait that distinguishes it from cluster_level and touch_level below.

### Parameters

In [ ]:
# pivot_level: select the detector and set its knobs (shared knobs come from BASE_LEVEL_PARAMS).
CFG_PIVOT = dataclasses.replace(
    BASE_LEVEL_PARAMS,
    level_detector="pivot_level",
    level_use_pullback=True,           # pivot_level is the only detector with a pullback family
)
CFG_PIVOT

### Detect

In [ ]:
levels_pivot = detect_levels(df, CFG_PIVOT)
print_level_counts(levels_pivot)

### Inspect

In [ ]:
levels_table(levels_pivot).head(20)

### Visualize

In [ ]:
plot_levels(df, levels_pivot, show_invalidated=False,
            title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | pivot_level").show()

### Levels with EMAs

In [ ]:
plot_levels_emas(levels_pivot,
                 f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | pivot_level + EMAs").show()

### Live signals

Auto-refreshing browser chart for this detector. It prints a clickable link and rewrites an HTML file under data/live each poll. Plots only - no orders. Interrupt the cell to stop. Works best with a rolling window (a num_candles spec) so the window advances.

In [ ]:
run_live_levels(CFG_PIVOT)

## Cluster level

Merges nearby confirmed pivots into one level per price zone (within cluster_merge_atr_mult times ATR of an active level). A level strengthens on each merge and dies only on a decisive close-through break - a close beyond it by cluster_break_atr_mult times ATR - so a wick poking the level does not kill it. The level price is fixed at its seeding pivot (no future-dependent drift), and strength counts how reinforced it is. Resistance and support only (no pullback). cluster_max_levels caps the simultaneously-active levels.

### Parameters

In [ ]:
# cluster_level: select the detector and set its knobs (shared knobs come from BASE_LEVEL_PARAMS).
CFG_CLUSTER = dataclasses.replace(
    BASE_LEVEL_PARAMS,
    level_detector="cluster_level",
    cluster_merge_atr_mult=0.5,        # merge pivots within this x ATR
    cluster_break_atr_mult=0.1,        # close must clear the level by this x ATR to break it
)
CFG_CLUSTER

### Detect

In [ ]:
levels_cluster = detect_levels(df, CFG_CLUSTER)
print_level_counts(levels_cluster)

### Inspect

In [ ]:
levels_table(levels_cluster).head(20)

### Visualize

In [ ]:
plot_levels(df, levels_cluster, show_invalidated=False,
            title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | cluster_level").show()

### Levels with EMAs

In [ ]:
plot_levels_emas(levels_cluster,
                 f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | cluster_level + EMAs").show()

### Live signals

Auto-refreshing browser chart for this detector. It prints a clickable link and rewrites an HTML file under data/live each poll. Plots only - no orders. Interrupt the cell to stop. Works best with a rolling window (a num_candles spec) so the window advances.

In [ ]:
run_live_levels(CFG_CLUSTER)

## Touch level

A price zone becomes a level only once it has been touched touch_min_touches times - significance by frequency rather than a single pivot. Nearby swings cluster into one zone (touch_cluster_mult), any bar coming within touch_band_mult counts as a touch, and a level appears the bar its count first reaches the threshold (so the filter is causal). touch_recency_bars, when above zero, drops a level that goes that many bars without a touch; zero keeps all. Resistance and support only.

### Parameters

In [ ]:
# touch_level: select the detector and set its knobs (shared knobs come from BASE_LEVEL_PARAMS).
CFG_TOUCH = dataclasses.replace(
    BASE_LEVEL_PARAMS,
    level_detector="touch_level",
    touch_min_touches=3,               # touches before a zone becomes a level
    touch_recency_bars=0,              # 0 = keep all; else drop levels gone stale
)
CFG_TOUCH

### Detect

In [ ]:
levels_touch = detect_levels(df, CFG_TOUCH)
print_level_counts(levels_touch)

### Inspect

In [ ]:
levels_table(levels_touch).head(20)

### Visualize

In [ ]:
plot_levels(df, levels_touch, show_invalidated=False,
            title=f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | touch_level").show()

### Levels with EMAs

In [ ]:
plot_levels_emas(levels_touch,
                 f"{DATA_CONFIG.symbol} {DATA_CONFIG.interval}m | touch_level + EMAs").show()

### Live signals

Auto-refreshing browser chart for this detector. It prints a clickable link and rewrites an HTML file under data/live each poll. Plots only - no orders. Interrupt the cell to stop. Works best with a rolling window (a num_candles spec) so the window advances.

In [ ]:
run_live_levels(CFG_TOUCH)